In [1]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.1 MB/s eta 0:00:00


In [ ]:
import os

source_path = "/kaggle/input/datasets/abcxyzi/raddet-icassp-2025/"

!rm -rf /kaggle/working/nist512_data
!mkdir -p /kaggle/working/nist512_data

print("Extracting NIST 512 parts aa through af...")

!cat {source_path}NISTSpecMaxHold512Data.tar.part-a* | tar -xf - -C /kaggle/working/nist512_data/

print("\nExtraction complete. Checking contents...")
!ls -F /kaggle/working/nist512_data/

Extracting NIST 512 parts aa through af...

Extraction complete. Checking contents...
NISTSpecMaxHold512Data/


In [ ]:
import yaml
import os

base_dir = None
for root, dirs, files in os.walk('/kaggle/working/nist512_data'):
    if 'images' in dirs:
        if os.path.exists(os.path.join(root, 'images', 'train')):
            base_dir = root
            break

if not base_dir:
    print("❌ Dataset structure not found. Please check Cell 1 output.")
else:
    print(f"✅ Dataset found at: {base_dir}")
    
    val_subdir = 'val'
    if os.path.exists(os.path.join(base_dir, 'images', 'valid')):
        val_subdir = 'valid'

    config = {
        'path': base_dir,
        'train': 'images/train',
        'val': f'images/{val_subdir}',
        'names': {
            0: 'P0N#1', 
            1: 'P0N#2', 
            2: 'Q3N#1', 
            3: 'Q3N#2', 
            4: 'Q3N#3'
        }
    }

    with open('/kaggle/working/nist512.yaml', 'w') as f:
        yaml.dump(config, f)
    
    print(f"✅ YAML created at /kaggle/working/nist512.yaml")

✅ Dataset found at: /kaggle/working/nist512_data/NISTSpecMaxHold512Data
✅ YAML created at /kaggle/working/nist512.yaml


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/kaggle/working/nist512.yaml',
    epochs=50,
    imgsz=512,
    batch=16,
    device=0,
    name='nist_512_small_model'
)

model.export(format='onnx')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/nist512.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz

'/kaggle/working/runs/detect/nist_512_small_model/weights/best.onnx'